# PBMC classification baseline

Audience:
- Researchers who want a simple supervised baseline on a familiar single-cell dataset.

Prerequisites:
- Install `scdlkit[tutorials]`.
- PBMC data available through Scanpy or the repository cache.

Learning goals:
- Train `mlp_classifier` on PBMC labels.
- Inspect accuracy and macro F1.
- Plot and save a confusion matrix.
- Keep the same notebook path on CPU or GPU with `device="auto"`.

Install:
```bash
python -m pip install "scdlkit[tutorials]"
```


## Outline

1. Load PBMC data with Scanpy.
2. Detect the runtime device.
3. Choose the notebook profile.
4. Fit the classification baseline with `device="auto"`.
5. Evaluate and save a report.
6. Plot a confusion matrix.


In [ ]:
from __future__ import annotations

from pathlib import Path

import scanpy as sc
import torch
from IPython.display import display

from scdlkit import TaskRunner

DATA_PATH = Path("examples/data/pbmc3k_processed.h5ad")
OUTPUT_DIR = Path("artifacts/pbmc_classification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_name}")


In [ ]:
TUTORIAL_PROFILE = "quickstart"  # change to "full" for a longer run

PROFILE = {
    "quickstart": {"epochs": 15, "batch_size": 128},
    "full": {"epochs": 30, "batch_size": 128},
}[TUTORIAL_PROFILE]

print(f"Tutorial profile: {TUTORIAL_PROFILE}")
print(PROFILE)


In [ ]:
adata = sc.read_h5ad(DATA_PATH) if DATA_PATH.exists() else sc.datasets.pbmc3k_processed()
print(adata)
print("Classification target:", "louvain")


## Train the classifier

This is the same code path on CPU and GPU. The runner will select CUDA automatically when it is available.

Treat this notebook as a baseline classifier, not a production-ready cell-annotation system. Its job is to tell you whether a simple supervised MLP is already competitive before you build something more specialized.


In [ ]:
runner = TaskRunner(
    model="mlp_classifier",
    task="classification",
    epochs=PROFILE["epochs"],
    batch_size=PROFILE["batch_size"],
    label_key="louvain",
    device="auto",
    output_dir=str(OUTPUT_DIR),
)
runner.fit(adata)
metrics = runner.evaluate()
metrics


## Save a report and inspect the confusion matrix

The notebook writes its report to `artifacts/pbmc_classification/`.


In [ ]:
runner.save_report(OUTPUT_DIR / "report.md")
loss_fig, _ = runner.plot_losses()
loss_fig.savefig(OUTPUT_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
confusion_fig, _ = runner.plot_confusion_matrix()
confusion_fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
display(confusion_fig)


## Expected outputs

After running the notebook, check:

- `accuracy`
- `macro_f1`
- `artifacts/pbmc_classification/report.md`
- `artifacts/pbmc_classification/loss_curve.png`
- `artifacts/pbmc_classification/confusion_matrix.png`

If you want a stronger baseline before interpreting the results, switch the first config cell to `TUTORIAL_PROFILE = "full"` and rerun the notebook.
